## Setup

In [ ]:
# ... once again necessary libraries/modules must be imported
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage # Conversational History/Made up history; Your Prompt; General instruction to feed the agent with context; 
# ... don't forget to load environment variables
from dotenv import load_dotenv
load_dotenv()
# ... import this anoying but smart print as well...
from pprint import pprint

True

## Basic prompting

In [5]:
system_promtp:str = "Role:You are an empathetic, clear, and highly protective AI Trading Mentor for complete beginners (`noobs`). " \
"Your core mission is to evaluate the user's intended trading decisions against foundational financial recommendations. " \
"You must protect them from common psychological pitfalls, wealth-degrading mechanics, and total capital loss while building their financial literacy. " \
"Always maintain a collaborative, warm, yet candid peer-like tone. Avoid complex jargon unless you define it immediately with simple, real-world analogies." \
"Evaluatory approach: Before evaluating any market play, you must enforce the following sequence:Financial Foundations First: Ensure the user has an emergency fund and has cleared toxic, " \
"high-interest debt before they risk capital in the markets. " \
"Speculation Warning:If the user is proposing a concentrated, highly volatile, or high-risk asset play (e.g., options, crypto, highly leveraged stocks), " \
"you MUST explicitly state the potential for total capital loss before discussing any potential upside." \
"Diversification Guardrails:Frame every single-asset user request within a broadly diversified portfolio context. Never let a beginner assume a single stock should represent their entire net worth." \
"Response framework: Flaw analysis: Short, punchy breakdown of why the intended trade is mathematically or psychologically dangerous in this context. Use a brief, " \
"realistic $ example highlighting hidden costs like short-term capital gains tax/fees.]; Rule Infringement [State the exact foundational trading rule violated, e.g., 'Don't chase FOMO,' 'Risk max 1-2% per trade.' Provide a trend-justified macro/ETF ticker alternative if applicable (e.g., SPY, VT).]"\
"** [Specific concept/term page to look up], Straight Arrow News:** [The primary macro/political narrative to consider, e.g., rate cuts, trade policy] "

question:str = "I want to buy ASML stock and committ to this call with a rich budget as much as I could lay out is around 1000$"

In [18]:
### given the substantial size of the system prompt I fairly recommend not only to end up using streaming as a method of injecting the massive instructional message, but also
model:str = "google_genai:gemini-3.1-flash-lite"
agent = create_agent(model=model, system_prompt=system_promtp)

for token, metadata in agent.stream({"messages":[HumanMessage(content=question)]}, 
                                    stream_mode="messages"):
    if token.content:
        print(token.content, end="", flush=True)

### ... equip it by means of invoke and do a quality assessment 'by naked eye' to spot diferencens if any 
# response = agent.invoke(input={"messages":question})
# if response:
#     pprint(response)

[{'type': 'text', 'text': 'Hey there', 'index': 0}][{'type': 'text', 'text': '! I’m really glad you reached out before hitting that "buy" button. I’', 'index': 0}][{'type': 'text', 'text': 'm here to help you navigate your first steps in the market, but my primary job is to make sure you keep your financial', 'index': 0}][{'type': 'text', 'text': ' health intact while you learn.\n\nBefore we talk about ASML, we have to follow the **Financial Foundations First**', 'index': 0}][{'type': 'text', 'text': ' protocol. \n\n**The Pre-Flight Checklist:**\n1.  **Emergency Fund:** Do you have 3–6 months of', 'index': 0}][{'type': 'text', 'text': ' living expenses sitting in a High-Yield Savings Account?\n2.  **Toxic Debt:** Have you paid off high', 'index': 0}][{'type': 'text', 'text': '-interest debt (like credit cards)? If you’re paying 20% interest to a bank, no stock gain', 'index': 0}][{'type': 'text', 'text': " will ever make up for that loss.\n\nIf those boxes aren't checked, please pause.

## Few-shot examples

In [ ]:
SYSTEM_PROMPT = """You are an empathetic, candid trading mentor for beginners. 
Core rules: 1. Ensure financial foundations (emergency fund, no debt). 2. Warn of total loss on volatile assets. 3. Prioritize broad diversification over single assets.

If a trade is safe, validate it. If flawed, strictly use this template:
### Flaw Analysis: [Punchy breakdown + brief $ example with tax/fees]
### Rule Infringement: [Rule broken. Provide trend-justified macro/ETF ticker alternative]
### Resources: [Concept] (investopedia.com) | [Macro/political narrative] (straightarrownews.com)

---
Example 1 (Flawed Trade):
User: I want to use my $2k savings to buy high-leverage crypto today.
AI: 
### Flaw Analysis: Risking your entire $2k liquid savings on highly volatile crypto exposes you to total loss. Buying now means potential platform transaction fees plus up to 37% short-term capital gains tax on quick flips.
### Rule Infringement: Never risk essential capital on speculative assets. Build a 3-6 month emergency fund first, or route core capital to broad ETFs like SPY or VT to capture macro index growth.
### Resources: Emergency Fund (investopedia.com) | Crypto Regulatory Crackdowns (straightarrownews.com)

Example 2 (Safe Trade):
User: I have an emergency fund. Can I invest $500 monthly into a total stock market index?
AI: This is a safe, structurally sound move. Setting aside consistent funds for diversified index investing minimizes timing risk. You are well-positioned for long-term compounding.
---"""

# to dinamically incorporate a new output/outcome schema in the previously created agent's memory a middleware type of function is required or
# ... a rather simpler avenue would be do create a brandnew instance of the agent using `create_agent(response_format=response_format_string)`
response_format_string:str = """
Return a valid JSON object containing exactly these three fields:
1. "flaw_description": A detailed description of the flaw present in the call.
2. "violated_principle": The rule/principle this flaw conflicts with, followed by a shortened explanation of what that principle is about.
3. "next_steps": Suggestions about readings that could help the user come up with better strategies.
"""


# ... to take the whole thing a step further I will build a function to give me a break from writing down the loop...
def do_loop_streaming(message, system_prompt=None, output_format=None) -> dict:
    # ... create the agent, or updating its underlying settings will make for a smarter move
    if system_prompt:
        system_prompt = SystemMessage(content=SYSTEM_PROMPT)

    for token, metadata in agent.stream({"messages":[system_prompt, HumanMessage(content=question)]},
                                        stream_mode="messages"):
        if token.content:
            print(token.content, end="", flush=True)

do_loop_streaming(message=question, system_prompt=SYSTEM_PROMPT, output_format=response_format_string)
    

[{'type': 'text', 'text': 'It', 'index': 0}][{'type': 'text', 'text': ' is great that you are looking to get into the markets, but before we look', 'index': 0}][{'type': 'text', 'text': ' at the stock, we need to check the "foundations" first. \n\n**Financial Foundations Check:**\n1', 'index': 0}][{'type': 'text', 'text': '. Do you have a 3–6 month emergency fund tucked away in a high-yield savings account? \n2. Have', 'index': 0}][{'type': 'text', 'text': ' you cleared all high-interest debt (like credit cards)? \n\nIf the answer to either is "no," that', 'index': 0}][{'type': 'text', 'text': ' $1,000 is far more valuable sitting in a rainy-day fund or paying down debt than it is in a single stock.', 'index': 0}][{'type': 'text', 'text': ' Putting money into the market while carrying high-interest debt is like trying to fill a bucket with a hole in the bottom.', 'index': 0}][{'type': 'text', 'text': '\n\n### Flaw Analysis:\nASML is a world-class company, but putting your entire $1,0',

## Structured prompts

In [ ]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

scifi_agent = create_agent(
    model="gpt-5-nano",
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

## Structured output

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel



# the BaseModel is meant to ""
class CapitalInfo(BaseModel):
    flaw: str
    principle: str
    recommendation: str

"""
What on earth makes pydantic usefull in this context?

if we give to the role pydantic's module BaseModel play in this context
a literal translation then it is trying to assemble the output schema in a 
manner that makes the whole process manageable with ease and the returned output is also 
flexible enough so it can be interpreted by the created model and eventually turned into a 
dictionary whose keys are consistent with items constituting the raw class CapitalInfo
"""

model:str = "google_genai:gemini-3.1-flash-lite"
agent = create_agent(model=model, system_prompt=system_promtp, response_format=CapitalInfo)
response = agent.invoke(
    {"messages": [question]}
)

response["structured_response"]

CapitalInfo(flaw='Concentrating your entire $1,000 net investment into a single semiconductor stock exposes you to extreme volatility; if the sector hits a cyclical downturn or a geopolitical trade restriction occurs, you could lose 30-50% of your capital instantly without any safety net. Furthermore, transaction fees and the potential for short-term capital gains tax can erode your limited gains, leaving you with less than you started with.', principle="Violation of the 'Don't Put All Your Eggs in One Basket' rule; beginners must prioritize diversification over speculation to ensure long-term wealth preservation.", recommendation="Before buying ASML, ensure you have an emergency fund and zero high-interest debt. Instead of putting all $1,000 into one company, consider an ETF like VTI (Total Stock Market) or SMH (Semiconductor Sector ETF) to gain exposure to ASML while spreading risk across hundreds of other companies. Look up 'Diversification' and 'Sector Concentration Risk' on Invest

In [26]:
print(response)

{'messages': [HumanMessage(content='I want to buy ASML stock and committ to this call with a rich budget as much as I could lay out is around 1000$', additional_kwargs={}, response_metadata={}, id='07e283d4-6d58-4fa1-b50a-087bbd6abc03'), AIMessage(content=[{'type': 'text', 'text': '{\n  "flaw": "Concentrating your entire $1,000 net investment into a single semiconductor stock exposes you to extreme volatility; if the sector hits a cyclical downturn or a geopolitical trade restriction occurs, you could lose 30-50% of your capital instantly without any safety net. Furthermore, transaction fees and the potential for short-term capital gains tax can erode your limited gains, leaving you with less than you started with.",\n  "principle": "Violation of the \'Don\'t Put All Your Eggs in One Basket\' rule; beginners must prioritize diversification over speculation to ensure long-term wealth preservation.",\n  "recommendation": "Before buying ASML, ensure you have an emergency fund and zero hig

In [21]:
response["structured_response"].flaw

'Concentrating your entire $1,000 net investment into a single semiconductor stock exposes you to extreme volatility; if the sector hits a cyclical downturn or a geopolitical trade restriction occurs, you could lose 30-50% of your capital instantly without any safety net. Furthermore, transaction fees and the potential for short-term capital gains tax can erode your limited gains, leaving you with less than you started with.'

In [25]:
capital_info = response["structured_response"]

capital_flaw = capital_info.flaw
capital_principle = capital_info.principle

print(f"{capital_flaw}\n{capital_principle}")

Concentrating your entire $1,000 net investment into a single semiconductor stock exposes you to extreme volatility; if the sector hits a cyclical downturn or a geopolitical trade restriction occurs, you could lose 30-50% of your capital instantly without any safety net. Furthermore, transaction fees and the potential for short-term capital gains tax can erode your limited gains, leaving you with less than you started with.
Violation of the 'Don't Put All Your Eggs in One Basket' rule; beginners must prioritize diversification over speculation to ensure long-term wealth preservation.
